In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.00),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   150.00),
]

columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.show()            # print the table
# df.printSchema()   # show column names + types
# df.count()         # count rows (returns a number)

# --- Summing a column ---
df.agg(F.sum("balance")).show()                    # total of all balances

# --- Sum with a WHERE clause ---
(df.filter(F.col("balance") > 1000.00)
   .agg(F.sum("balance"))
   .show())                                        # total of balances over 1000

# --- Sum with two conditions (AND) ---
(df.filter((F.col("balance") > 1000.00) & (F.col("account_type") == "Savings"))
   .agg(F.sum("balance"))
   .show())                                        # Savings balances over 1000

# --- Filter rows where the name contains a lowercase "e" ---
(df.filter((F.col("balance") > 1000.00) & (F.col("name").contains("e")))
   .agg(F.sum("balance"))
   .show())                                        # over-1000 with an "e" in the name

# --- Just show matching rows instead of summing ---
(df.filter(F.col("name").contains("e"))
   .show())                                        # people with an "e" in their name


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.select("name", "balance").show() 


(df.withColumn("with_interest", F.col("balance") * 1.05)
   .show())


(df.withColumn("tier",
        F.when(F.col("balance") >= 5000, "Gold")
         .when(F.col("balance") >= 1000, "Silver")
         .otherwise("Bronze"))
   .show())


df.select("name", "balance").show() 

df.withColumn("balance_rounded", F.round( F.col("balance"),0)).show()

df.withColumn("active", F.when (F.col("balance") > 0, "Yes")
              .otherwise("No")).show()


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  12750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

(df.groupBy("account_type")
   .count()
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .show())


(df.groupBy("account_type")
   .agg(
       F.count("*").alias("num_accounts"),
       F.sum("balance").alias("total_balance"),
       F.round(F.avg("balance"),2).alias("avg_balance"),
       F.min("balance").alias("min_balance"),
       F.max("balance").alias("max_balance"),
   )
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 2000)
   .show())


(df.groupBy("account_type")
   .count()
   .show())



(df.groupBy("account_type")
   .agg(
       F.round (F.avg("balance"),2).alias("Avg_balance"),
       F.round(F.sum("balance"),2).alias("total_balance"),
       F.max("balance").alias("max_balance")
   )
   .show())

(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 10000)
   .show())

In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with categories (like our 'tier' column earlier)
data = [
    ("Alice", "Gold", 6000),
    ("Bob", "Silver", 2500),
    ("Charlie", "Bronze", 500),
    ("Diana", "Gold", 8000),
    ("Evan", "Silver", 1500),
    ("Fiona", "Bronze", 800)
]

columns = ["name", "tier", "balance"]
df = spark.createDataFrame(data, columns)

# 2. Group by tier and calculate metrics (Total balance, Average balance, and Customer count)
summary_df = df.groupBy("tier").agg(
    F.sum("balance").alias("total_balance"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.count("name").alias("customer_count")
)

# 3. Show the resulting aggregated DataFrame
summary_df.show()


from pyspark.sql import functions as F

# 1. Group by tier, filter for groups with more than 1 customer, and sort by total balance descending
filtered_summary_df = (
    df.groupBy("tier")
    .agg(
        F.sum("balance").alias("total_balance"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.count("name").alias("customer_count")
    )
    .filter(F.col("avg_balance") >= 1000)
    .orderBy(F.col("total_balance").asc())
)

filtered_summary_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create a customer accounts DataFrame
accounts_data = [
    (101, "Alice", "Gold"),
    (102, "Bob", "Silver"),
    (103, "Charlie", "Bronze"),
    (104, "Diana", "Gold")
]
accounts_df = spark.createDataFrame(accounts_data, ["account_id", "name", "tier"])

# 2. Create a separate transactions DataFrame
transactions_data = [
    (101, 500),
    (101, 1200),
    (102, 300),
    (105, 999) # ID 105 doesn't exist in accounts
]
transactions_df = spark.createDataFrame(transactions_data, ["account_id", "transaction_amount"])

# 3. Perform a Left Join to keep all accounts and match their transactions
joined_df = accounts_df.join(
    transactions_df, 
    on="account_id", 
    how="left"
)

joined_df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create sample transaction data
data = [
    (101, "2026-09-01", 100),
    (101, "2026-09-05", 250),
    (101, "2026-09-10", 75),
    (102, "2026-09-02", 500),
    (102, "2026-09-08", 150)
]
df = spark.createDataFrame(data, ["account_id", "date", "amount"])

# 2. Define a window specification: partition by account, order by date
window_spec = Window.partitionBy("account_id").orderBy("date")

# 3. Apply window functions (Row Number and Running Total / Cumulative Sum)
ranked_df = df.withColumn("transaction_seq", F.row_number().over(window_spec)) \
              .withColumn("running_total", F.sum("amount").over(window_spec))

ranked_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with missing values (None represents NULL in Python)
data = [
    (101, "Alice", 5000, "Active"),
    (102, "Bob", None, "Pending"),
    (103, None, 1200, None),
    (104, "Diana", 3400, "Active")
]
df = spark.createDataFrame(data, ["account_id", "name", "balance", "status"])

# 2. Fill missing balances with 0 and missing names/status with defaults
cleaned_df = df.na.fill({"balance": 0.0, "name": "Unknown Customer", "status": "Inactive"})

# 3. Use Coalesce to create a fallback priority column
fallback_df = cleaned_df.withColumn(
    "display_status", 
    F.coalesce(F.col("status"), F.lit("Default Status"))
)

fallback_df.show()

In [0]:
from pyspark.sql import functions as F

data = [
    (101, "Alice", None),
    (102, "Bob", "Pending")
]
df = spark.createDataFrame(data, ["account_id", "name", "status"])

# Coalesce checks 'status' first; if it's null, it falls back to the literal string
result_df = df.withColumn(
    "display_status", 
    F.coalesce(F.col("status"), F.lit("Default Status"))
)

result_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create a small DataFrame to save
data = [(101, "Alice", 5000), (102, "Bob", 2500)]
df = spark.createDataFrame(data, ["account_id", "name", "balance"])

# 2. Write the DataFrame to a managed Delta table or file path
# (Using a temporary managed table name for easy testing in Databricks)
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("customer_balances_summary")
)

# 3. Read it right back to verify it was saved successfully
saved_df = spark.table("customer_balances_summary")
saved_df.show()

In [0]:
%sql
SELECT * FROM customer_balances_summary WHERE balance > 2000;

In [0]:
from pyspark.sql import functions as F

# 1. Create sample account data
data = [
    (101, "Active", 5000.0),
    (102, "Pending", 1500.0),
    (103, "Active", 3200.0),
    (104, "Inactive", 400.0),
    (105, "Active", 7500.0)
]
df = spark.createDataFrame(data, ["account_id", "status", "balance"])

# 2. Group by status and calculate total balance, average balance, and account count
agg_df = df.groupBy("status").agg(
    F.sum("balance").alias("total_balance"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.count("account_id").alias("account_count")
)




agg_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create customer DataFrame
customers_data = [
    (101, "Alice", "London","UK"),
    (102, "Bob", "Birmingham","France"),
    (103, "Charlie", "Manchester","Germany"),
    (104, "Diana", "Paris","Sweden")
]
customers_df = spark.createDataFrame(customers_data, ["account_id", "name", "city", "country"])

# 2. Create transaction summary DataFrame
transactions_data = [
    (101, 5000.0,"UK"),
    (102, 1500.0,"France"),
    (104, 3200.0,"Sweden")
]
transactions_df = spark.createDataFrame(transactions_data, ["account_id",  "balance", "country"])

joined_df = customers_df.join(
    transactions_df , 
    on=["account_id", "country"], 
    how="inner"  #left, right, full
)
 

joined_df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create sample transaction data
data = [
    (1, "London", 500.0),
    (2, "London", 1200.0),
    (3, "London", 800.0),
    (4, "Birmingham", 300.0),
    (5, "Birmingham", 1500.0)
]
df = spark.createDataFrame(data, ["txn_id", "branch", "amount"])

# 2. Define the window specification: partition by branch, order by amount descending - this does not hold data, 
#.   but is blueprint or the partitioniing used in the following winfoes function.
window_spec = Window.partitionBy("branch").orderBy(F.col("amount").desc())

# 3. Apply the row_number function
ranked_df = df.withColumn("rank_in_branch", F.row_number().over(window_spec))

ranked_df.show()


In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with missing values (None / null)
data = [
    (101, "Alice", 5000.0),
    (102, None, 1500.0),
    (103, "Charlie", None),
    (104, "Diana", 0.0)
]
df = spark.createDataFrame(data, ["account_id", "name", "balance"])

# 2. Fill missing names with 'Unknown' and missing balances with 0.0
cleaned_df = df.fillna({
    "name": "Unknown",
    "balance": 0.0
})

cleaned_df.show()